# 容量约束车辆路径问题(CVRP)

**类别：** 路径

来源：[https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-cvrp](https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-cvrp)


## 问题描述

**在容量约束车辆路径问题(Capacitated Vehicle Routing Problem, CVRP)**中,一组具有相同容量的配送车辆必须为具有单一商品已知需求的客户提供服务。车辆从一个共同的配送中心出发并返回。每个客户必须恰好由一辆车服务,且每辆车服务的客户总需求不得超过其容量。目标是最小化总行驶距离,同时最小化所使用的车辆数量。

	

### 学习要点

- 添加 [list decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 以建模每辆卡车的客户序列
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算行驶距离
- 添加[多目标](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html)


## 数据

所提供的容量约束车辆路径问题(CVRP)算例来自 [Augerat 等人的 Set A 数据集](http://neo.lcc.uma.es/vrp/vrp-instances/capacitated-vrp-instances/)。它们遵循 [TSPLib 格式](http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/DOC.PS):

- 节点数量由关键字 DIMENSION 指定(其中包括一个配送中心,因此客户数量为节点数减 1)。
- 卡车容量由关键字 CAPACITY 指定。
- 边的类型由 EDGE_WEIGHT_TYPE 指定。请注意,在我们的模型中仅接受 EUC_2D 这一种边类型。
- 在关键字 NODE_COORD_SECTION 之后:每个节点的 ID 以及其 x、y 坐标。
- 在关键字 DEMAND_SECTION 之后:每个节点的 ID 及其需求。
- 配送中心列在关键字 DEPOT_SECTION 之后。请注意,在我们的模型中仅接受一个配送中心。

可用车辆数量等于客户数量。


## 建模方法

容量约束车辆路径问题(CVRP)的 Hexaly 模型使用 list decision variables。对每辆卡车,我们定义一个列表变量,表示其所访问的客户序列。通过对所有列表施加 **partition** 约束,我们确保每个客户恰好由一辆卡车服务。

当一辆卡车至少访问一个客户时,它才被车队所使用。借助 **count** 算子(返回列表中的元素数量),我们可以检查每辆卡车是否被使用,从而计算车队中使用的卡车总数。

我们可以使用需求数组上的 **at** 算子来访问序列中每个客户的需求。每辆卡车所配送的总量通过一个 lambda 函数计算,该函数将 `sum` 算子应用于所有被访问的客户。请注意,该 sum 中的项数以及列表的大小在搜索过程中会变化。

从一个客户到下一个客户所行驶的距离同样使用二维距离矩阵上的 **at** 算子来访问。我们使用另一个 [**lambda 函数**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 计算每辆卡车的总行驶距离,该函数对序列中相邻客户之间的距离求和。

两个目标以字典序方式定义。我们首先最小化所使用的卡车数量,然后最小化所有卡车的总行驶距离。


## 结果

**在容量约束车辆路径问题(CVRP)上,Hexaly 在最多包含 1,000 个客户的 CVRPLIB 基准上,在 1 分钟内相对于研究中的最佳已知解达到了 1.2% 的平均差距**。我们的[容量约束车辆路径(CVRP)基准测试页面](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-capacitated-vehicle-routing-problem-cvrp)给出了详细结果。

[查看该基准](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-capacitated-vehicle-routing-problem-cvrp)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math


def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


def main(instance_file, str_time_limit, output_file):
    #
    # Read instance data
    #
    nb_customers, nb_trucks, truck_capacity, dist_matrix_data, \
        dist_depot_data, demands_data = read_input_cvrp(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Sequence of customers visited by each truck
        customers_sequences = [model.list(nb_customers) for _ in range(nb_trucks)]

        # All customers must be visited by exactly one truck
        model.constraint(model.partition(customers_sequences))

        # Create Hexaly arrays to be able to access them with an "at" operator
        demands = model.array(demands_data)
        dist_matrix = model.array(dist_matrix_data)
        dist_depot = model.array(dist_depot_data)

        # A truck is used if it visits at least one customer
        trucks_used = [(model.count(customers_sequences[k]) > 0) for k in range(nb_trucks)]

        dist_routes = [None] * nb_trucks
        for k in range(nb_trucks):
            sequence = customers_sequences[k]
            c = model.count(sequence)

            # The quantity needed in each route must not exceed the truck capacity
            demand_lambda = model.lambda_function(lambda j: demands[j])
            route_quantity = model.sum(sequence, demand_lambda)
            model.constraint(route_quantity <= truck_capacity)

            # Distance traveled by each truck
            dist_lambda = model.lambda_function(lambda i:
                                                model.at(dist_matrix,
                                                         sequence[i - 1],
                                                         sequence[i]))
            dist_routes[k] = model.sum(model.range(1, c), dist_lambda) \
                + model.iif(c > 0,
                            dist_depot[sequence[0]] + dist_depot[sequence[c - 1]],
                            0)

        # Total number of trucks used
        nb_trucks_used = model.sum(trucks_used)

        # Total distance traveled
        total_distance = model.sum(dist_routes)

        # Objective: minimize the number of trucks used, then minimize the distance traveled
        model.minimize(nb_trucks_used)
        model.minimize(total_distance)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = int(str_time_limit)

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        #  - number of trucks used and total distance
        #  - for each truck the customers visited (omitting the start/end at the depot)
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d %d\n" % (nb_trucks_used.value, total_distance.value))
                for k in range(nb_trucks):
                    if trucks_used[k].value != 1:
                        continue
                    # Values in sequence are in 0...nbCustomers. +2 is to put it back
                    # in 2...nbCustomers+2 as in the data files (1 being the depot)
                    for customer in customers_sequences[k].value:
                        f.write("%d " % (customer + 2))
                    f.write("\n")


# The input files follow the "Augerat" format
def read_input_cvrp(filename):
    file_it = iter(read_elem(filename))

    nb_nodes = 0
    while True:
        token = next(file_it)
        if token == "DIMENSION":
            next(file_it)  # Removes the ":"
            nb_nodes = int(next(file_it))
            nb_customers = nb_nodes - 1
            nb_trucks = nb_customers
        elif token == "CAPACITY":
            next(file_it)  # Removes the ":"
            truck_capacity = int(next(file_it))
        elif token == "EDGE_WEIGHT_TYPE":
            next(file_it)  # Removes the ":"
            token = next(file_it)
            if token != "EUC_2D":
                print("Edge Weight Type " + token + " is not supported (only EUD_2D)")
                sys.exit(1)
        elif token == "NODE_COORD_SECTION":
            break

    customers_x = [None] * nb_customers
    customers_y = [None] * nb_customers
    depot_x = 0
    depot_y = 0
    for n in range(nb_nodes):
        node_id = int(next(file_it))
        if node_id != n + 1:
            print("Unexpected index")
            sys.exit(1)
        if node_id == 1:
            depot_x = int(next(file_it))
            depot_y = int(next(file_it))
        else:
            # -2 because original customer indices are in 2..nbNodes
            customers_x[node_id - 2] = int(next(file_it))
            customers_y[node_id - 2] = int(next(file_it))

    distance_matrix = compute_distance_matrix(customers_x, customers_y)
    distance_depots = compute_distance_depots(depot_x, depot_y, customers_x, customers_y)

    token = next(file_it)
    if token != "DEMAND_SECTION":
        print("Expected token DEMAND_SECTION")
        sys.exit(1)

    demands = [None] * nb_customers
    for n in range(nb_nodes):
        node_id = int(next(file_it))
        if node_id != n + 1:
            print("Unexpected index")
            sys.exit(1)
        if node_id == 1:
            if int(next(file_it)) != 0:
                print("Demand for depot should be 0")
                sys.exit(1)
        else:
            # -2 because original customer indices are in 2..nbNodes
            demands[node_id - 2] = int(next(file_it))

    token = next(file_it)
    if token != "DEPOT_SECTION":
        print("Expected token DEPOT_SECTION")
        sys.exit(1)

    depot_id = int(next(file_it))
    if depot_id != 1:
        print("Depot id is supposed to be 1")
        sys.exit(1)

    end_of_depot_section = int(next(file_it))
    if end_of_depot_section != -1:
        print("Expecting only one depot, more than one found")
        sys.exit(1)

    return nb_customers, nb_trucks, truck_capacity, distance_matrix, \
        distance_depots, demands


# Compute the distance matrix
def compute_distance_matrix(customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_matrix = [[None for i in range(nb_customers)] for j in range(nb_customers)]
    for i in range(nb_customers):
        distance_matrix[i][i] = 0
        for j in range(nb_customers):
            dist = compute_dist(customers_x[i], customers_x[j], customers_y[i], customers_y[j])
            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist
    return distance_matrix


# Compute the distances to depot
def compute_distance_depots(depot_x, depot_y, customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_depots = [None] * nb_customers
    for i in range(nb_customers):
        dist = compute_dist(depot_x, customers_x[i], depot_y, customers_y[i])
        distance_depots[i] = dist
    return distance_depots


def compute_dist(xi, xj, yi, yj):
    exact_dist = math.sqrt(math.pow(xi - xj, 2) + math.pow(yi - yj, 2))
    return int(math.floor(exact_dist + 0.5))


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python cvrp.py input_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) > 2 else None
    str_time_limit = sys.argv[3] if len(sys.argv) > 3 else "20"

    main(instance_file, str_time_limit, output_file)
